In [1]:
import sympy as sp
import numpy as np
import math
import collections
import tqdm
import itertools
import random

import utils

In [2]:
N_POINTS = 4

conj = lambda x: x.subs(sp.I, -sp.I)


def as_complex(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0] * sp.I + c[1]


def as_tuple(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0], c[1]


ss = sp.var("".join([f"s{i}," for i in range(N_POINTS)]))
ts = sp.var("".join([f"t{i}," for i in range(N_POINTS)]))
ps = [s + sp.I * t for s, t in zip(ss, ts)]


# first pair
A0 = as_tuple(ps[0] * ps[1] * ps[2] * ps[3])
A1 = as_tuple(ps[0] * ps[1] * conj(ps[2]) * conj(ps[3]))

# second pair
A2 = as_tuple(ps[0] * conj(ps[1]) * ps[2] * ps[3])
A3 = as_tuple(ps[0] * ps[1] * conj(ps[2]) * ps[3])


# Same output, which is expected since all P related shifts are cancelled when we subtract.
Q12 = (A1[0] * A1[1]) ** 2 + (A2[0] * A2[1]) ** 2
Q13 = (A1[0] * A1[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q10 = (A1[0] * A1[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q23 = (A2[0] * A2[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q20 = (A2[0] * A2[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q30 = (A3[0] * A3[1]) ** 2 + (A0[0] * A0[1]) ** 2

D1 = (Q12 - Q30).as_poly()
D2 = (Q13 - Q20).as_poly()
D3 = (Q10 - Q23).as_poly()

In [3]:
f = D1.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 - 2*s0**4*s1**3*s2**4*s3**3*t1*t3 + 2*s0**4*s1**3*s2**4*s3*t1*t3**3 - s0**4*s1**3*s2**3*s3**4*t1*t2 + 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 - s0**4*s1**3*s2**3*t1*t2*t3**4 + 12*s0**4*s1**3*s2**2*s3**3*t1*t2**2*t3 - 12*s0**4*s1**3*s2**2*s3*t1*t2**2*t3**3 + s0**4*s1**3*s2*s3**4*t1*t2**3 - 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 + s0**4*s1**3*s2*t1*t2**3*t3**4 - 2*s0**4*s1**3*s3**3*t1*t2**4*t3 + 2*s0**4*s1**3*s3*t1*t2**4*t3**3 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 + 2*s0**4*s1*s2**4*s3**3*t1**3*t3 - 2*s0**4*s1*s2**4*s3*t1**3*t3**3 + s0**4*s1*s2**3*s3**4*t1**3*t2 - 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 + s0**4*s1*s2**3*t1**3*t2*t3**4 - 12*s0**4*s1*s2**2*s3**3*t1**3*t2**2*t3 + 12*s0**4*s1*s2**2*s3*t1**3*t2**2*t3**3 - s0**4*s1*s2*s3**4*t1

In [4]:
f = D2.factor_list()
f

(-4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - 6*s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 + s0**3*s1**4*s2**4*s3**3*t0*t3 - s0**3*s1**4*s2**4*s3*t0*t3**3 + 2*s0**3*s1**4*s2**3*s3**4*t0*t2 - 12*s0**3*s1**4*s2**3*s3**2*t0*t2*t3**2 + 2*s0**3*s1**4*s2

In [5]:
f = D3.factor_list()
f

(4,
 [(Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - 6*s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + 6*s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - 6*s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + 6*s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + 6*s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - 6*s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + 6*s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - 6*s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 - s0**3*s1**4*s2**4*s3**3*t0*t3 + s0**3*s1**4*s2**4*s3*t0*t3**3 + 6*s0**3*s1**4*s2**2*s3**3*t0*t2**2*t3 - 6*s0**3*s1**4*s2**2*s3*t0*t2**2*t3**3 - s0**3*s1**4*

In [6]:
print((D1 - D2).is_zero, (D1 - D3).is_zero, (D2 - D3).is_zero)
print((D1 + D2).is_zero, (D1 + D3).is_zero, (D2 + D3).is_zero)

False False False
False False False


In [7]:
modulus = 5
D3_ = (
    (
        (D3 // 4).subs(
            [
                (s ** (modulus + i), s ** ((modulus + i) % (modulus - 1)))
                for s in ss
                for i in range(10)
            ]
            + [
                (t ** (modulus + i), t ** ((modulus + i) % (modulus - 1)))
                for t in ts
                for i in range(10)
            ]
        )
    )
    .as_poly()
    .set_modulus(modulus)
)
D3_

Poly(s0**4*s1**4*s2**3*s3**3*t2*t3 - s0**4*s1**4*s2**3*s3*t2*t3**3 - s0**4*s1**4*s2*s3**3*t2**3*t3 + s0**4*s1**4*s2*s3*t2**3*t3**3 + s0**4*s1**3*s2**3*s3**4*t1*t2 - s0**4*s1**3*s2**3*s3**2*t1*t2*t3**2 + s0**4*s1**3*s2**3*t1*t2*t3**4 - s0**4*s1**3*s2*s3**4*t1*t2**3 + s0**4*s1**3*s2*s3**2*t1*t2**3*t3**2 - s0**4*s1**3*s2*t1*t2**3*t3**4 - s0**4*s1**2*s2**3*s3**3*t1**2*t2*t3 + s0**4*s1**2*s2**3*s3*t1**2*t2*t3**3 + s0**4*s1**2*s2*s3**3*t1**2*t2**3*t3 - s0**4*s1**2*s2*s3*t1**2*t2**3*t3**3 - s0**4*s1*s2**3*s3**4*t1**3*t2 + s0**4*s1*s2**3*s3**2*t1**3*t2*t3**2 - s0**4*s1*s2**3*t1**3*t2*t3**4 + s0**4*s1*s2*s3**4*t1**3*t2**3 - s0**4*s1*s2*s3**2*t1**3*t2**3*t3**2 + s0**4*s1*s2*t1**3*t2**3*t3**4 + s0**4*s2**3*s3**3*t1**4*t2*t3 - s0**4*s2**3*s3*t1**4*t2*t3**3 - s0**4*s2*s3**3*t1**4*t2**3*t3 + s0**4*s2*s3*t1**4*t2**3*t3**3 - s0**3*s1**4*s2**4*s3**3*t0*t3 + s0**3*s1**4*s2**4*s3*t0*t3**3 + s0**3*s1**4*s2**2*s3**3*t0*t2**2*t3 - s0**3*s1**4*s2**2*s3*t0*t2**2*t3**3 - s0**3*s1**4*s3**3*t0*t2**4*t3 + s0**3*s

In [8]:
# r = list(sp.groebner([D1, A1[0] + A1[1] + A2[0] + A2[1] - A3[0] - A0[0] - A0[1]]))
# r = list(sp.groebner([D2, A1[0] - A3[0] - A2[0] + A0[0]]))
# r = list(sp.groebner([D3, A1[0] - A0[0] - A2[0] + A3[0]]))
# r

In [9]:
# p0 = r[0].as_poly()
# p0
# [p.as_poly().total_degree() for p in r]

# The overcomplicated approach

But currently there is no simpler thing that is viable

In [23]:
N_POINTS = 8

conj = lambda x: x.subs(sp.I, -sp.I)


def as_complex(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0] * sp.I + c[1]


def as_tuple(x):
    c = x.as_poly(sp.I).coeffs()
    return c[0], c[1]


ss = sp.var("".join([f"s{i}," for i in range(N_POINTS)]))
ts = sp.var("".join([f"t{i}," for i in range(N_POINTS)]))
ps = [s + sp.I * t for s, t in zip(ss, ts)]

# ps[0] = sp.Integer(1)

ps[1] = sp.Integer(1)
# ps[2] = sp.Integer(1)
# ps[4] = sp.Integer(1)

# ps[3] = sp.Integer(1)
# ps[5] = sp.Integer(1)
ps[6] = sp.Integer(1)

ps[7] = sp.Integer(1)


A0 = as_tuple(ps[0] * ps[1] * ps[2] * ps[3] * ps[4] * ps[5] * ps[6] * ps[7])
A1 = as_tuple(
    ps[0]
    * conj(ps[1])
    * ps[2]
    * conj(ps[3])
    * ps[4]
    * conj(ps[5])
    * ps[6]
    * conj(ps[7])
)
A2 = as_tuple(
    ps[0]
    * ps[1]
    * conj(ps[2])
    * conj(ps[3])
    * ps[4]
    * ps[5]
    * conj(ps[6])
    * conj(ps[7])
)
A3 = as_tuple(
    ps[0]
    * ps[1]
    * ps[2]
    * ps[3]
    * conj(ps[4])
    * conj(ps[5])
    * conj(ps[6])
    * conj(ps[7])
)


# Same output, which is expected since all P related shifts are cancelled when we subtract.
Q12 = (A1[0] * A1[1]) ** 2 + (A2[0] * A2[1]) ** 2
Q13 = (A1[0] * A1[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q10 = (A1[0] * A1[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q23 = (A2[0] * A2[1]) ** 2 + (A3[0] * A3[1]) ** 2
Q20 = (A2[0] * A2[1]) ** 2 + (A0[0] * A0[1]) ** 2
Q30 = (A3[0] * A3[1]) ** 2 + (A0[0] * A0[1]) ** 2

D1 = (Q12 - Q30).as_poly()
D2 = (Q13 - Q20).as_poly()
D3 = (Q10 - Q23).as_poly()

KeyboardInterrupt: 

In [18]:
f = D1.factor_list()
f
# D1

(-4,
 [(Poly(s0**4*s3**4*s4**3*s6**3*t4*t6 - s0**4*s3**4*s4**3*s6*t4*t6**3 - s0**4*s3**4*s4*s6**3*t4**3*t6 + s0**4*s3**4*s4*s6*t4**3*t6**3 + s0**4*s3**3*s4**3*s6**4*t3*t4 - 6*s0**4*s3**3*s4**3*s6**2*t3*t4*t6**2 + s0**4*s3**3*s4**3*t3*t4*t6**4 - s0**4*s3**3*s4*s6**4*t3*t4**3 + 6*s0**4*s3**3*s4*s6**2*t3*t4**3*t6**2 - s0**4*s3**3*s4*t3*t4**3*t6**4 - 6*s0**4*s3**2*s4**3*s6**3*t3**2*t4*t6 + 6*s0**4*s3**2*s4**3*s6*t3**2*t4*t6**3 + 6*s0**4*s3**2*s4*s6**3*t3**2*t4**3*t6 - 6*s0**4*s3**2*s4*s6*t3**2*t4**3*t6**3 - s0**4*s3*s4**3*s6**4*t3**3*t4 + 6*s0**4*s3*s4**3*s6**2*t3**3*t4*t6**2 - s0**4*s3*s4**3*t3**3*t4*t6**4 + s0**4*s3*s4*s6**4*t3**3*t4**3 - 6*s0**4*s3*s4*s6**2*t3**3*t4**3*t6**2 + s0**4*s3*s4*t3**3*t4**3*t6**4 + s0**4*s4**3*s6**3*t3**4*t4*t6 - s0**4*s4**3*s6*t3**4*t4*t6**3 - s0**4*s4*s6**3*t3**4*t4**3*t6 + s0**4*s4*s6*t3**4*t4**3*t6**3 - s0**3*s3**4*s4**3*s6**4*t0*t4 + 6*s0**3*s3**4*s4**3*s6**2*t0*t4*t6**2 - s0**3*s3**4*s4**3*t0*t4*t6**4 + s0**3*s3**4*s4*s6**4*t0*t4**3 - 6*s0**3*s3**4*s4*s6

In [19]:
f = D2.factor_list()
f

(4,
 [(Poly(s0**4*s3**4*s4**3*s6**3*t4*t6 - s0**4*s3**4*s4**3*s6*t4*t6**3 - s0**4*s3**4*s4*s6**3*t4**3*t6 + s0**4*s3**4*s4*s6*t4**3*t6**3 - 2*s0**4*s3**3*s4**4*s6**3*t3*t6 + 2*s0**4*s3**3*s4**4*s6*t3*t6**3 - s0**4*s3**3*s4**3*s6**4*t3*t4 + 6*s0**4*s3**3*s4**3*s6**2*t3*t4*t6**2 - s0**4*s3**3*s4**3*t3*t4*t6**4 + 12*s0**4*s3**3*s4**2*s6**3*t3*t4**2*t6 - 12*s0**4*s3**3*s4**2*s6*t3*t4**2*t6**3 + s0**4*s3**3*s4*s6**4*t3*t4**3 - 6*s0**4*s3**3*s4*s6**2*t3*t4**3*t6**2 + s0**4*s3**3*s4*t3*t4**3*t6**4 - 2*s0**4*s3**3*s6**3*t3*t4**4*t6 + 2*s0**4*s3**3*s6*t3*t4**4*t6**3 - 6*s0**4*s3**2*s4**3*s6**3*t3**2*t4*t6 + 6*s0**4*s3**2*s4**3*s6*t3**2*t4*t6**3 + 6*s0**4*s3**2*s4*s6**3*t3**2*t4**3*t6 - 6*s0**4*s3**2*s4*s6*t3**2*t4**3*t6**3 + 2*s0**4*s3*s4**4*s6**3*t3**3*t6 - 2*s0**4*s3*s4**4*s6*t3**3*t6**3 + s0**4*s3*s4**3*s6**4*t3**3*t4 - 6*s0**4*s3*s4**3*s6**2*t3**3*t4*t6**2 + s0**4*s3*s4**3*t3**3*t4*t6**4 - 12*s0**4*s3*s4**2*s6**3*t3**3*t4**2*t6 + 12*s0**4*s3*s4**2*s6*t3**3*t4**2*t6**3 - s0**4*s3*s4*s6**4*t3

In [20]:
f = D3.factor_list()
f

(4,
 [(Poly(s0**4*s3**4*s4**3*s6**3*t4*t6 - s0**4*s3**4*s4**3*s6*t4*t6**3 - s0**4*s3**4*s4*s6**3*t4**3*t6 + s0**4*s3**4*s4*s6*t4**3*t6**3 + s0**4*s3**3*s4**3*s6**4*t3*t4 - 6*s0**4*s3**3*s4**3*s6**2*t3*t4*t6**2 + s0**4*s3**3*s4**3*t3*t4*t6**4 - s0**4*s3**3*s4*s6**4*t3*t4**3 + 6*s0**4*s3**3*s4*s6**2*t3*t4**3*t6**2 - s0**4*s3**3*s4*t3*t4**3*t6**4 - 6*s0**4*s3**2*s4**3*s6**3*t3**2*t4*t6 + 6*s0**4*s3**2*s4**3*s6*t3**2*t4*t6**3 + 6*s0**4*s3**2*s4*s6**3*t3**2*t4**3*t6 - 6*s0**4*s3**2*s4*s6*t3**2*t4**3*t6**3 - s0**4*s3*s4**3*s6**4*t3**3*t4 + 6*s0**4*s3*s4**3*s6**2*t3**3*t4*t6**2 - s0**4*s3*s4**3*t3**3*t4*t6**4 + s0**4*s3*s4*s6**4*t3**3*t4**3 - 6*s0**4*s3*s4*s6**2*t3**3*t4**3*t6**2 + s0**4*s3*s4*t3**3*t4**3*t6**4 + s0**4*s4**3*s6**3*t3**4*t4*t6 - s0**4*s4**3*s6*t3**4*t4*t6**3 - s0**4*s4*s6**3*t3**4*t4**3*t6 + s0**4*s4*s6*t3**4*t4**3*t6**3 + 2*s0**3*s3**4*s4**4*s6**3*t0*t6 - 2*s0**3*s3**4*s4**4*s6*t0*t6**3 + s0**3*s3**4*s4**3*s6**4*t0*t4 - 6*s0**3*s3**4*s4**3*s6**2*t0*t4*t6**2 + s0**3*s3**4*s4**

In [21]:
print((D1 - D2).is_zero, (D1 - D3).is_zero, (D2 - D3).is_zero)
print((D1 + D2).is_zero, (D1 + D3).is_zero, (D2 + D3).is_zero)

False False False
False False False


Poly(-4*s0**4*s2**4*s6**3*s7**3*t6*t7 + 4*s0**4*s2**4*s6**3*s7*t6*t7**3 + 4*s0**4*s2**4*s6*s7**3*t6**3*t7 - 4*s0**4*s2**4*s6*s7*t6**3*t7**3 + 4*s0**4*s2**3*s6**3*s7**4*t2*t6 - 24*s0**4*s2**3*s6**3*s7**2*t2*t6*t7**2 + 4*s0**4*s2**3*s6**3*t2*t6*t7**4 - 4*s0**4*s2**3*s6*s7**4*t2*t6**3 + 24*s0**4*s2**3*s6*s7**2*t2*t6**3*t7**2 - 4*s0**4*s2**3*s6*t2*t6**3*t7**4 + 24*s0**4*s2**2*s6**3*s7**3*t2**2*t6*t7 - 24*s0**4*s2**2*s6**3*s7*t2**2*t6*t7**3 - 24*s0**4*s2**2*s6*s7**3*t2**2*t6**3*t7 + 24*s0**4*s2**2*s6*s7*t2**2*t6**3*t7**3 - 4*s0**4*s2*s6**3*s7**4*t2**3*t6 + 24*s0**4*s2*s6**3*s7**2*t2**3*t6*t7**2 - 4*s0**4*s2*s6**3*t2**3*t6*t7**4 + 4*s0**4*s2*s6*s7**4*t2**3*t6**3 - 24*s0**4*s2*s6*s7**2*t2**3*t6**3*t7**2 + 4*s0**4*s2*s6*t2**3*t6**3*t7**4 - 4*s0**4*s6**3*s7**3*t2**4*t6*t7 + 4*s0**4*s6**3*s7*t2**4*t6*t7**3 + 4*s0**4*s6*s7**3*t2**4*t6**3*t7 - 4*s0**4*s6*s7*t2**4*t6**3*t7**3 - 4*s0**3*s2**4*s6**4*s7**3*t0*t7 + 4*s0**3*s2**4*s6**4*s7*t0*t7**3 + 24*s0**3*s2**4*s6**2*s7**3*t0*t6**2*t7 - 24*s0**3*s2**

In [22]:
D1

Poly(-4*s0**4*s3**4*s4**3*s6**3*t4*t6 + 4*s0**4*s3**4*s4**3*s6*t4*t6**3 + 4*s0**4*s3**4*s4*s6**3*t4**3*t6 - 4*s0**4*s3**4*s4*s6*t4**3*t6**3 - 4*s0**4*s3**3*s4**3*s6**4*t3*t4 + 24*s0**4*s3**3*s4**3*s6**2*t3*t4*t6**2 - 4*s0**4*s3**3*s4**3*t3*t4*t6**4 + 4*s0**4*s3**3*s4*s6**4*t3*t4**3 - 24*s0**4*s3**3*s4*s6**2*t3*t4**3*t6**2 + 4*s0**4*s3**3*s4*t3*t4**3*t6**4 + 24*s0**4*s3**2*s4**3*s6**3*t3**2*t4*t6 - 24*s0**4*s3**2*s4**3*s6*t3**2*t4*t6**3 - 24*s0**4*s3**2*s4*s6**3*t3**2*t4**3*t6 + 24*s0**4*s3**2*s4*s6*t3**2*t4**3*t6**3 + 4*s0**4*s3*s4**3*s6**4*t3**3*t4 - 24*s0**4*s3*s4**3*s6**2*t3**3*t4*t6**2 + 4*s0**4*s3*s4**3*t3**3*t4*t6**4 - 4*s0**4*s3*s4*s6**4*t3**3*t4**3 + 24*s0**4*s3*s4*s6**2*t3**3*t4**3*t6**2 - 4*s0**4*s3*s4*t3**3*t4**3*t6**4 - 4*s0**4*s4**3*s6**3*t3**4*t4*t6 + 4*s0**4*s4**3*s6*t3**4*t4*t6**3 + 4*s0**4*s4*s6**3*t3**4*t4**3*t6 - 4*s0**4*s4*s6*t3**4*t4**3*t6**3 + 4*s0**3*s3**4*s4**3*s6**4*t0*t4 - 24*s0**3*s3**4*s4**3*s6**2*t0*t4*t6**2 + 4*s0**3*s3**4*s4**3*t0*t4*t6**4 - 4*s0**3*s3**4